# 04 — Feature Engineering

**Objective:** Transform data into a customer-level feature dataset with rolling windows, RFM features, trend signals, and risk indicators. Prevent data leakage.

---

In [1]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

from src.config import *
from src.utils import save_csv, normalize_minmax

pd.set_option('display.max_columns', None)

In [2]:
# Load cleaned data
loyalty = pd.read_csv(CLEANED_DATA_DIR / 'loyalty_with_churn.csv')
flights = pd.read_csv(CLEANED_DATA_DIR / 'flights_cleaned.csv')

flights['YearMonth'] = pd.to_datetime(flights['YearMonth'])
loyalty['Enrollment_Date'] = pd.to_datetime(loyalty['Enrollment_Date'])

print(f'Loyalty: {loyalty.shape}  |  Flights: {flights.shape}')

# Define observation cutoff — features are built BEFORE this; churn label is AFTER this
max_date = flights['YearMonth'].max()
cutoff_date = max_date - pd.DateOffset(months=CHURN_WINDOW_MONTHS)
print(f'\nFeature window: everything up to {cutoff_date.strftime("%Y-%m")}')
print(f'Churn window: {cutoff_date.strftime("%Y-%m")} to {max_date.strftime("%Y-%m")}')

Loyalty: (16737, 25)  |  Flights: (389065, 9)

Feature window: everything up to 2018-06
Churn window: 2018-06 to 2018-12


## 1. Feature Window Data

We split the flight data into:
- **Feature window**: All months BEFORE the cutoff (used for building features)
- **Churn window**: Last 6 months (used ONLY for the churn label — already computed)

In [3]:
# Split into feature window and churn window
feature_flights = flights[flights['YearMonth'] <= cutoff_date].copy()
print(f'Feature window flights: {feature_flights.shape[0]:,} records')
print(f'Date range: {feature_flights["YearMonth"].min()} to {feature_flights["YearMonth"].max()}')

Feature window flights: 288,643 records
Date range: 2017-01-01 00:00:00 to 2018-06-01 00:00:00


## 2. Recency Features

How recently did the customer engage?

In [5]:
# Last activity dates per customer
last_flight = feature_flights[feature_flights['Total Flights'] > 0].groupby(PK)['YearMonth'].max().reset_index()
last_flight.columns = [PK, 'Last_Flight_Date']

last_earning = feature_flights[feature_flights['Points Accumulated'] > 0].groupby(PK)['YearMonth'].max().reset_index()
last_earning.columns = [PK, 'Last_Earning_Date']

last_redemption = feature_flights[feature_flights['Points Redeemed'] > 0].groupby(PK)['YearMonth'].max().reset_index()
last_redemption.columns = [PK, 'Last_Redemption_Date']

# Calculate days since last activity (relative to cutoff date)
recency = loyalty[[PK]].copy()
recency = recency.merge(last_flight, on=PK, how='left')
recency = recency.merge(last_earning, on=PK, how='left')
recency = recency.merge(last_redemption, on=PK, how='left')

recency['Days_Since_Last_Flight'] = (cutoff_date - recency['Last_Flight_Date']).dt.days.fillna(9999)
recency['Days_Since_Last_Earning'] = (cutoff_date - recency['Last_Earning_Date']).dt.days.fillna(9999)
recency['Days_Since_Last_Redemption'] = (cutoff_date - recency['Last_Redemption_Date']).dt.days.fillna(9999)

# Cap at reasonable max (3 years = 1095 days)
for col in ['Days_Since_Last_Flight', 'Days_Since_Last_Earning', 'Days_Since_Last_Redemption']:
    recency[col] = recency[col].clip(upper=1095)

recency = recency[[PK, 'Days_Since_Last_Flight', 'Days_Since_Last_Earning', 'Days_Since_Last_Redemption']]
print('Recency features:')
print(recency.describe().T[['mean', 'std', 'min', 'max']])

Recency features:
                                     mean            std       min       max
Loyalty Number              549735.880445  258912.132453  100018.0  999986.0
Days_Since_Last_Flight         208.627114     399.942001       0.0    1095.0
Days_Since_Last_Earning        208.627114     399.942001       0.0    1095.0
Days_Since_Last_Redemption     572.186115     463.757768       0.0    1095.0


## 3. Frequency Features

How often does the customer fly?

In [6]:
def rolling_agg(df, pk, date_col, value_col, agg_func, window_months, cutoff, col_name):
    """Compute rolling aggregation for a given window."""
    window_start = cutoff - pd.DateOffset(months=window_months)
    window_data = df[(df[date_col] > window_start) & (df[date_col] <= cutoff)]
    result = window_data.groupby(pk)[value_col].agg(agg_func).reset_index()
    result.columns = [pk, col_name]
    return result

# Frequency features
frequency = loyalty[[PK]].copy()

for window in ROLLING_WINDOWS:
    freq_feat = rolling_agg(feature_flights, PK, 'YearMonth', 'Total Flights', 'sum', window, cutoff_date, f'Flights_{window}M')
    frequency = frequency.merge(freq_feat, on=PK, how='left')

# Lifetime flights
lifetime_flights = feature_flights.groupby(PK)['Total Flights'].sum().reset_index()
lifetime_flights.columns = [PK, 'Flights_Lifetime']
frequency = frequency.merge(lifetime_flights, on=PK, how='left')

# Active months count
active_months = feature_flights[feature_flights['Total Flights'] > 0].groupby(PK)['YearMonth'].nunique().reset_index()
active_months.columns = [PK, 'Active_Months']
frequency = frequency.merge(active_months, on=PK, how='left')

# Fill NaN with 0
frequency = frequency.fillna(0)

print('Frequency features:')
print(frequency.describe().T[['mean', 'std', 'min', 'max']])

Frequency features:
                           mean            std       min       max
Loyalty Number    549735.880445  258912.132453  100018.0  999986.0
Flights_3M             4.130489       3.563604       0.0      20.0
Flights_6M             6.910079       4.915440       0.0      28.0
Flights_12M           14.114298       9.051862       0.0      53.0
Flights_Lifetime      20.236124      12.857741       0.0      74.0
Active_Months          7.633985       4.697848       0.0      18.0


## 4. Monetary Features

Points, distance, and dollar value.

In [7]:
monetary = loyalty[[PK]].copy()

for window in ROLLING_WINDOWS:
    # Points earned
    pts_earned = rolling_agg(feature_flights, PK, 'YearMonth', 'Points Accumulated', 'sum', window, cutoff_date, f'Points_Earned_{window}M')
    monetary = monetary.merge(pts_earned, on=PK, how='left')
    
    # Points redeemed
    pts_redeemed = rolling_agg(feature_flights, PK, 'YearMonth', 'Points Redeemed', 'sum', window, cutoff_date, f'Points_Redeemed_{window}M')
    monetary = monetary.merge(pts_redeemed, on=PK, how='left')
    
    # Distance
    dist = rolling_agg(feature_flights, PK, 'YearMonth', 'Distance', 'sum', window, cutoff_date, f'Distance_{window}M')
    monetary = monetary.merge(dist, on=PK, how='left')

# Lifetime totals
lifetime_monetary = feature_flights.groupby(PK).agg(
    Points_Earned_Lifetime=('Points Accumulated', 'sum'),
    Points_Redeemed_Lifetime=('Points Redeemed', 'sum'),
    Distance_Lifetime=('Distance', 'sum'),
    Dollar_Cost_Lifetime=('Dollar Cost Points Redeemed', 'sum'),
).reset_index()

monetary = monetary.merge(lifetime_monetary, on=PK, how='left')
monetary = monetary.fillna(0)

print('Monetary features:')
print(monetary.describe().T[['mean', 'std', 'min', 'max']])

Monetary features:
                                   mean            std       min       max
Loyalty Number            549735.880445  258912.132453  100018.0  999986.0
Points_Earned_3M            6450.821205    6265.003428       0.0   40442.0
Points_Redeemed_3M            98.917727     226.817446       0.0    1862.0
Distance_3M                 6304.901715    5974.458422       0.0   40442.0
Points_Earned_6M           10684.500986    8252.650178       0.0   52386.0
Points_Redeemed_6M           181.649101     309.174802       0.0    2229.0
Distance_6M                10483.261516    7963.885322       0.0   44699.0
Points_Earned_12M          21455.059330   14091.263595       0.0   81938.0
Points_Redeemed_12M          356.270180     462.780306       0.0    3435.0
Distance_12M               21238.437474   14053.506035       0.0   81938.0
Points_Earned_Lifetime     30616.528052   19639.813730       0.0  118817.0
Points_Redeemed_Lifetime     509.719842     579.610201       0.0    3969.0
Distan

## 5. Trend Features

Is the customer's activity increasing or decreasing?

In [8]:
def compute_trend(df, pk, date_col, value_col, window_months, cutoff, col_name):
    """Compute linear trend (slope) over a rolling window."""
    window_start = cutoff - pd.DateOffset(months=window_months)
    window_data = df[(df[date_col] > window_start) & (df[date_col] <= cutoff)].copy()
    
    # Create month index (0, 1, 2, ...)
    window_data['month_idx'] = ((window_data[date_col] - window_start).dt.days / 30.44).astype(int)
    
    def get_slope(group):
        if len(group) < 2:
            return 0.0
        try:
            slope, _, _, _, _ = stats.linregress(group['month_idx'], group[value_col])
            return slope
        except:
            return 0.0
    
    trends = window_data.groupby(pk).apply(get_slope).reset_index()
    trends.columns = [pk, col_name]
    return trends

# Compute 6-month trends
trend = loyalty[[PK]].copy()

flight_trend = compute_trend(feature_flights, PK, 'YearMonth', 'Total Flights', 6, cutoff_date, 'Flight_Trend_6M')
trend = trend.merge(flight_trend, on=PK, how='left')

points_trend = compute_trend(feature_flights, PK, 'YearMonth', 'Points Accumulated', 6, cutoff_date, 'Points_Trend_6M')
trend = trend.merge(points_trend, on=PK, how='left')

distance_trend = compute_trend(feature_flights, PK, 'YearMonth', 'Distance', 6, cutoff_date, 'Distance_Trend_6M')
trend = trend.merge(distance_trend, on=PK, how='left')

# Engagement trend: combined flights + points trend
trend['Engagement_Trend'] = (trend['Flight_Trend_6M'].fillna(0) + 
                              normalize_minmax(trend['Points_Trend_6M'].fillna(0))) / 2

trend = trend.fillna(0)
print('Trend features:')
print(trend.describe().T[['mean', 'std', 'min', 'max']])

Trend features:
                            mean            std            min            max
Loyalty Number     549735.880445  258912.132453  100018.000000  999986.000000
Flight_Trend_6M         0.234471       0.545597      -1.294118       2.294118
Points_Trend_6M       380.766926     995.637735   -3002.382353    4917.415385
Distance_Trend_6M     372.773246     960.480431   -2381.815385    4917.415385
Engagement_Trend        0.330823       0.331261      -0.569416       1.644381


## 6. Loyalty Features

Tier, tenure, enrollment characteristics.

In [9]:
loyalty_features = loyalty[[PK, 'Loyalty_Card_Ordinal', 'Tenure_Months', 
                            'Education_Ordinal', 'Salary', 'CLV',
                            'Salary_Missing']].copy()

# Enrollment age (months from enrollment to cutoff)
loyalty_features['Enrollment_Age_Months'] = (
    (cutoff_date - loyalty['Enrollment_Date']).dt.days / 30.44
).round(0).astype(int)

# Enrollment type encoding
loyalty_features['Is_Promo_Enrollment'] = (loyalty['Enrollment Type'] == '2018 Promotion').astype(int)

# Gender encoding
loyalty_features['Is_Male'] = (loyalty['Gender'] == 'Male').astype(int)

# Marital status encoding
loyalty_features['Is_Married'] = (loyalty['Marital Status'] == 'Married').astype(int)
loyalty_features['Is_Single'] = (loyalty['Marital Status'] == 'Single').astype(int)

print('Loyalty features:')
print(loyalty_features.describe().T[['mean', 'std', 'min', 'max']])

Loyalty features:
                                mean            std          min          max
Loyalty Number         549735.880445  258912.132453  100018.0000  999986.0000
Loyalty_Card_Ordinal        0.748581       0.773301       0.0000       2.0000
Tenure_Months              35.451933      24.384892       0.0000      80.0000
Education_Ordinal           1.771405       0.770489       0.0000       4.0000
Salary                  77660.037223   27999.491403   43788.0000  239867.0000
CLV                      7895.655488    6317.864027    2230.4528   35928.6432
Salary_Missing              0.253211       0.434864       0.0000       1.0000
Enrollment_Age_Months      32.292346      23.605926      -6.0000      74.0000
Is_Promo_Enrollment         0.058015       0.233779       0.0000       1.0000
Is_Male                     0.497520       0.500009       0.0000       1.0000
Is_Married                  0.581645       0.493304       0.0000       1.0000
Is_Single                   0.267909       0.4

## 7. Behavioral Features

Redemption patterns, consistency, and engagement depth.

In [10]:
behavioral = loyalty[[PK]].copy()

# Redemption ratio (lifetime points redeemed / points earned)
redemption_data = feature_flights.groupby(PK).agg(
    total_earned=('Points Accumulated', 'sum'),
    total_redeemed=('Points Redeemed', 'sum'),
    total_flights=('Total Flights', 'sum'),
    total_distance=('Distance', 'sum'),
    total_months=('YearMonth', 'nunique'),
).reset_index()

redemption_data['Redemption_Ratio'] = np.where(
    redemption_data['total_earned'] > 0,
    redemption_data['total_redeemed'] / redemption_data['total_earned'],
    0
)

# Average distance per flight
redemption_data['Avg_Distance_Per_Flight'] = np.where(
    redemption_data['total_flights'] > 0,
    redemption_data['total_distance'] / redemption_data['total_flights'],
    0
)

# Activity consistency (coefficient of variation of monthly flights)
monthly_flights = feature_flights.groupby([PK, 'YearMonth'])['Total Flights'].sum().reset_index()
flight_cv = monthly_flights.groupby(PK)['Total Flights'].agg(['mean', 'std']).reset_index()
flight_cv['Activity_Consistency'] = np.where(
    flight_cv['mean'] > 0,
    1 - (flight_cv['std'].fillna(0) / flight_cv['mean']).clip(upper=3) / 3,
    0
)

behavioral = behavioral.merge(redemption_data[[PK, 'Redemption_Ratio', 'Avg_Distance_Per_Flight']], on=PK, how='left')
behavioral = behavioral.merge(flight_cv[[PK, 'Activity_Consistency']], on=PK, how='left')

# Seasonal preference (which quarter has most flights?)
feature_flights['Quarter'] = feature_flights['YearMonth'].dt.quarter
quarterly_flights = feature_flights.groupby([PK, 'Quarter'])['Total Flights'].sum().reset_index()
preferred_quarter = quarterly_flights.loc[quarterly_flights.groupby(PK)['Total Flights'].idxmax()]
preferred_quarter = preferred_quarter[[PK, 'Quarter']].rename(columns={'Quarter': 'Preferred_Quarter'})
behavioral = behavioral.merge(preferred_quarter, on=PK, how='left')

behavioral = behavioral.fillna(0)
print('Behavioral features:')
print(behavioral.describe().T[['mean', 'std', 'min', 'max']])

Behavioral features:
                                  mean            std       min            max
Loyalty Number           549735.880445  258912.132453  100018.0  999986.000000
Redemption_Ratio              0.014966       0.022379       0.0       0.743590
Avg_Distance_Per_Flight    1262.444571     605.572981       0.0    2496.000000
Activity_Consistency          0.474102       0.260155       0.0       0.862868
Preferred_Quarter             1.773317       0.773839       1.0       4.000000


## 8. Risk Signal Features

Early warning indicators of churn.

In [11]:
risk = loyalty[[PK]].copy()

# Declining activity: flights in last 3M < flights in prior 3M
flights_recent_3m = rolling_agg(feature_flights, PK, 'YearMonth', 'Total Flights', 'sum', 3, cutoff_date, 'flights_recent_3m')
prior_cutoff = cutoff_date - pd.DateOffset(months=3)
flights_prior_3m = rolling_agg(feature_flights, PK, 'YearMonth', 'Total Flights', 'sum', 3, prior_cutoff, 'flights_prior_3m')

risk = risk.merge(flights_recent_3m, on=PK, how='left')
risk = risk.merge(flights_prior_3m, on=PK, how='left')
risk = risk.fillna(0)

risk['Declining_Activity'] = (risk['flights_recent_3m'] < risk['flights_prior_3m']).astype(int)

# Declining redemption
redeem_recent = rolling_agg(feature_flights, PK, 'YearMonth', 'Points Redeemed', 'sum', 3, cutoff_date, 'redeem_recent')
redeem_prior = rolling_agg(feature_flights, PK, 'YearMonth', 'Points Redeemed', 'sum', 3, prior_cutoff, 'redeem_prior')
risk = risk.merge(redeem_recent, on=PK, how='left')
risk = risk.merge(redeem_prior, on=PK, how='left')
risk = risk.fillna(0)
risk['Declining_Redemption'] = (risk['redeem_recent'] < risk['redeem_prior']).astype(int)

# Months inactive (consecutive months with 0 flights at end of feature window)
def months_inactive_at_end(group):
    sorted_group = group.sort_values('YearMonth', ascending=False)
    count = 0
    for _, row in sorted_group.iterrows():
        if row['Total Flights'] == 0:
            count += 1
        else:
            break
    return count

# This is expensive, so compute on the last 12 months only
last_12m_start = cutoff_date - pd.DateOffset(months=12)
last_12m = feature_flights[(feature_flights['YearMonth'] > last_12m_start) & (feature_flights['YearMonth'] <= cutoff_date)]
inactive_months = last_12m.groupby(PK).apply(months_inactive_at_end).reset_index()
inactive_months.columns = [PK, 'Months_Inactive']
risk = risk.merge(inactive_months, on=PK, how='left')
risk['Months_Inactive'] = risk['Months_Inactive'].fillna(12)  # No records = fully inactive

# Keep only final risk features
risk = risk[[PK, 'Declining_Activity', 'Declining_Redemption', 'Months_Inactive']]

print('Risk features:')
print(risk.describe().T[['mean', 'std', 'min', 'max']])

Risk features:
                               mean            std       min       max
Loyalty Number        549735.880445  258912.132453  100018.0  999986.0
Declining_Activity         0.245086       0.430151       0.0       1.0
Declining_Redemption       0.141722       0.348775       0.0       1.0
Months_Inactive            2.819263       4.483227       0.0      12.0


## 9. Combine All Features

In [12]:
# Merge all feature sets
customer_features = loyalty[[PK, 'Churn']].copy()

feature_dfs = [recency, frequency, monetary, trend, loyalty_features, behavioral, risk]

for df in feature_dfs:
    customer_features = customer_features.merge(df, on=PK, how='left')

# Drop the duplicate PK columns (if any)
customer_features = customer_features.loc[:, ~customer_features.columns.duplicated()]

print(f'\n=== FINAL FEATURE DATASET ===')
print(f'Shape: {customer_features.shape}')
print(f'\nFeature columns ({customer_features.shape[1] - 2} features + PK + target):')
for i, col in enumerate(customer_features.columns):
    print(f'  {i+1:3d}. {col}')

print(f'\nMissing values:\n{customer_features.isnull().sum()[customer_features.isnull().sum() > 0]}')

# Fill any remaining NaN
customer_features = customer_features.fillna(0)


=== FINAL FEATURE DATASET ===
Shape: (16737, 45)

Feature columns (43 features + PK + target):
    1. Loyalty Number
    2. Churn
    3. Days_Since_Last_Flight
    4. Days_Since_Last_Earning
    5. Days_Since_Last_Redemption
    6. Flights_3M
    7. Flights_6M
    8. Flights_12M
    9. Flights_Lifetime
   10. Active_Months
   11. Points_Earned_3M
   12. Points_Redeemed_3M
   13. Distance_3M
   14. Points_Earned_6M
   15. Points_Redeemed_6M
   16. Distance_6M
   17. Points_Earned_12M
   18. Points_Redeemed_12M
   19. Distance_12M
   20. Points_Earned_Lifetime
   21. Points_Redeemed_Lifetime
   22. Distance_Lifetime
   23. Dollar_Cost_Lifetime
   24. Flight_Trend_6M
   25. Points_Trend_6M
   26. Distance_Trend_6M
   27. Engagement_Trend
   28. Loyalty_Card_Ordinal
   29. Tenure_Months
   30. Education_Ordinal
   31. Salary
   32. CLV
   33. Salary_Missing
   34. Enrollment_Age_Months
   35. Is_Promo_Enrollment
   36. Is_Male
   37. Is_Married
   38. Is_Single
   39. Redemption_Ratio
   

In [13]:
# Feature summary statistics
feature_cols = [c for c in customer_features.columns if c not in [PK, 'Churn']]
print(f'Total features: {len(feature_cols)}')
customer_features[feature_cols].describe().T

Total features: 43


,count,mean,std,min,25%,50%,75%,max
Days_Since_Last_Flight,16737.0,208.627114,399.942001,0.000000,0.000000,31.000000,92.000000,1095.000000
Days_Since_Last_Earning,16737.0,208.627114,399.942001,0.000000,0.000000,31.000000,92.000000,1095.000000
Days_Since_Last_Redemption,16737.0,572.186115,463.757768,0.000000,120.000000,396.000000,1095.000000,1095.000000
Flights_3M,16737.0,4.130489,3.563604,0.000000,0.000000,4.000000,7.000000,20.000000
Flights_6M,16737.0,6.910079,4.915440,0.000000,3.000000,7.000000,10.000000,28.000000
Flights_12M,16737.0,14.114298,9.051862,0.000000,8.000000,15.000000,21.000000,53.000000
Flights_Lifetime,16737.0,20.236124,12.857741,0.000000,10.000000,23.000000,30.000000,74.000000
Active_Months,16737.0,7.633985,4.697848,0.000000,3.000000,9.000000,11.000000,18.000000
Points_Earned_3M,16737.0,6450.821205,6265.003428,0.000000,0.000000,5255.000000,10214.000000,40442.000000
Points_Redeemed_3M,16737.0,98.917727,226.817446,0.000000,0.000000,0.000000,0.000000,1862.000000


In [14]:
# Correlation with churn target
churn_corr = customer_features[feature_cols + ['Churn']].corr()['Churn'].drop('Churn').sort_values(ascending=False)
print('Top 15 features correlated with Churn (positive = more churn):')
print(churn_corr.head(15).to_string())
print('\nTop 15 features inversely correlated (negative = less churn):')
print(churn_corr.tail(15).to_string())

Top 15 features correlated with Churn (positive = more churn):
Months_Inactive               0.620591
Days_Since_Last_Earning       0.551463
Days_Since_Last_Flight        0.551463
Days_Since_Last_Redemption    0.335052
Loyalty_Card_Ordinal          0.011906
Education_Ordinal             0.008463
Is_Married                    0.008082
CLV                           0.005675
Salary                        0.000812
Is_Promo_Enrollment          -0.000065
Is_Single                    -0.001280
Is_Male                      -0.005471
Salary_Missing               -0.005701
Enrollment_Age_Months        -0.022499
Points_Trend_6M              -0.123389

Top 15 features inversely correlated (negative = less churn):
Points_Earned_3M          -0.340560
Distance_3M               -0.353331
Flights_3M                -0.395365
Points_Earned_6M          -0.428952
Distance_6M               -0.439377
Avg_Distance_Per_Flight   -0.444692
Distance_Lifetime         -0.470611
Points_Earned_Lifetime    -0.474259
F

## 10. Save Feature Dataset

In [15]:
save_csv(customer_features, FEATURES_DIR / 'customer_features.csv')

print(f'\nFeature Engineering Complete!')
print(f'Saved: customer_features.csv ({customer_features.shape})')
print(f'Features: {len(feature_cols)}')
print(f'Target: Churn (1={customer_features["Churn"].sum():,} churned, 0={(customer_features["Churn"]==0).sum():,} active)')
print(f'Churn rate: {customer_features["Churn"].mean()*100:.1f}%')

2026-06-11 23:02:36 | airline_loyalty | INFO | Saved: customer_features.csv (16,737 rows × 45 cols)



Feature Engineering Complete!
Saved: customer_features.csv ((16737, 45))
Features: 43
Target: Churn (1=2,794 churned, 0=13,943 active)
Churn rate: 16.7%
